In [1]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "requests", "pandas"], check=True)

import requests
import pandas as pd
import json




[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [4]:
ETHERSCAN_API_KEY = "MCFNNKXMI6AARMMQQXHAQH5A3HITP8NQJZ"
TEST_ADDRESS = "0xAb5801a7D398351b8bE11C439e05C5B3259aeC9B"

In [5]:
def get_deployed_contracts(deployer_address: str) -> list:
    url = "https://api.etherscan.io/v2/api"
    params = {
        "chainid": 1,
        "module": "account",
        "action": "txlist",
        "address": deployer_address,
        "startblock": 0,
        "endblock": 99999999,
        "sort": "asc",
        "apikey": ETHERSCAN_API_KEY
    }
    res = requests.get(url, params=params, timeout=10).json()
    print("Etherscan:", res.get("status"), "|", res.get("message"))
    if res.get("status") != "1":
        return []
    contracts = [
        {"address": tx["contractAddress"], "block": int(tx["blockNumber"]), "timestamp": int(tx["timeStamp"])}
        for tx in res["result"]
        if tx.get("to") == "" and tx.get("contractAddress")
    ]
    print(f"Deployments found: {len(contracts)}")
    return contracts


def check_sourcify_deep(contract_address: str, chain_id: int = 1) -> dict | None:
    url = f"https://sourcify.dev/server/v2/contract/{chain_id}/{contract_address}?fields=all"
    try:
        res = requests.get(url, timeout=10)
        if res.status_code != 200:
            return None
        d = res.json()

        # Deployment info
        deployment = d.get("deployment", {})
        block_number = deployment.get("blockNumber")
        verified_at = d.get("verifiedAt", "")

        # Match types — теперь ДВА отдельных поля
        creation_match = d.get("creationMatch", "")   # совпадение при создании
        runtime_match  = d.get("runtimeMatch", "")    # совпадение в рантайме
        overall_match  = d.get("match", "")           # итоговый

        # Proxy — продвинутый паттерн разработки
        proxy = d.get("proxyResolution", {})
        is_proxy = proxy.get("isProxy", False)

        # Документация — написал ли разработчик доки к коду
        has_userdoc = bool(d.get("userdoc", {}).get("methods"))
        has_devdoc  = bool(d.get("devdoc", {}).get("methods"))

        # Компилятор
        compilation = d.get("compilation", {})
        settings = compilation.get("compilerSettings", {})
        optimizer = settings.get("optimizer", {})

        # Исходный код — сколько файлов
        sources = d.get("sources", {})
        source_file_count = len(sources)

        return {
            "address": contract_address,
            # Матчи
            "overallMatch":    overall_match,    # "exact_match" или "match"
            "creationMatch":   creation_match,   # при деплое
            "runtimeMatch":    runtime_match,    # в рантайме
            # Контракт
            "name":            compilation.get("name"),
            "language":        compilation.get("language"),
            "compilerVersion": compilation.get("compilerVersion"),
            "evmVersion":      settings.get("evmVersion"),
            "optimizerEnabled":optimizer.get("enabled"),
            "optimizerRuns":   optimizer.get("runs"),  # сколько раз оптимизировал
            # Уникальные поля Sourcify
            "hasStorageLayout":bool(d.get("storageLayout")),
            "hasABI":          bool(d.get("abi")),
            "hasUserDoc":      has_userdoc,   # документация для пользователей
            "hasDevDoc":       has_devdoc,    # документация для разработчиков
            "sourceFileCount": source_file_count,  # сколько файлов кода
            "isProxy":         is_proxy,      # прокси-контракт (продвинуто)
            # Время
            "verifiedAt":      verified_at,
            "blockNumber":     block_number,
        }
    except Exception as e:
        print(f"Warning {contract_address[:10]}: {e}")
        return None


def audit_developer(deployer_address: str) -> dict:
    print(f"\nAuditing wallet: {deployer_address}\n")
    contracts = get_deployed_contracts(deployer_address)

    if not contracts:
        return {
            "deployer": deployer_address,
            "total_deployed": 0,
            "verified": 0,
            "reliability_score": 0.0,
            "verdict": "REJECT — No deployment history",
            "stats": {},
            "df": pd.DataFrame()
        }

    # Статистика по деплоям — даже до Sourcify
    blocks = [c["block"] for c in contracts]
    timestamps = [c["timestamp"] for c in contracts]
    deploy_span_days = (max(timestamps) - min(timestamps)) / 86400 if len(timestamps) > 1 else 0
    first_deploy = datetime.utcfromtimestamp(min(timestamps)).strftime("%Y-%m-%d")
    last_deploy  = datetime.utcfromtimestamp(max(timestamps)).strftime("%Y-%m-%d")

    results = []
    print("Checking on Sourcify...")
    for c in contracts[:15]:
        data = check_sourcify_deep(c["address"])
        if data:
            data["deployBlock"] = c["block"]
            results.append(data)
            print(f"  ✓ {c['address'][:12]}... | {data['overallMatch']} | "
                  f"{data['name']} | proxy:{data['isProxy']} | "
                  f"docs:{data['hasDevDoc']} | files:{data['sourceFileCount']}")
        time.sleep(0.15)

    df = pd.DataFrame(results) if results else pd.DataFrame()
    total = len(contracts)
    verified = len(results)

    # ── СКОРИНГ — каждый пункт объяснён ──────────────────────────────
    score = 0.0

    # 1. Есть вообще верифицированные контракты? (базовый порог)
    if verified > 0:
        score += 0.20

    # 2. Доля верифицированных от проверенных (насколько прозрачен)
    ratio = verified / min(total, 15)
    score += 0.10 * ratio

    # 3. exact_match (высший уровень верификации) — профессионал
    if not df.empty and (df["overallMatch"] == "exact_match").any():
        score += 0.15

    # 4. И creation И runtime матч — полная верификация
    if not df.empty and (df["creationMatch"].str.contains("match", na=False)).any():
        score += 0.05
    if not df.empty and (df["runtimeMatch"].str.contains("match", na=False)).any():
        score += 0.05

    # 5. Storage layout — полная документация структуры контракта
    if not df.empty and df["hasStorageLayout"].any():
        score += 0.10

    # 6. ABI — контракт задокументирован
    if not df.empty and df["hasABI"].any():
        score += 0.05

    # 7. DevDoc / UserDoc — написал документацию к коду (редкость!)
    if not df.empty and df["hasDevDoc"].any():
        score += 0.05
    if not df.empty and df["hasUserDoc"].any():
        score += 0.05

    # 8. Прокси-контракт — продвинутый паттерн, опытный разработчик
    if not df.empty and df["isProxy"].any():
        score += 0.05

    # 9. Несколько файлов исходного кода — большой проект
    if not df.empty and df["sourceFileCount"].max() > 3:
        score += 0.05

    # 10. Опыт по времени — деплоит давно и регулярно
    if deploy_span_days > 365:
        score += 0.05
    elif deploy_span_days > 90:
        score += 0.02

    # 11. Несколько языков (Solidity + Vyper) — широкий опыт
    if not df.empty and df["language"].nunique() > 1:
        score += 0.05

    score = round(min(score, 1.0), 2)

    if score >= 0.7:   verdict = "APPROVE — Open prediction market"
    elif score >= 0.4: verdict = "CONDITIONAL — Open with stricter KPI"
    else:              verdict = "REJECT — Low trust score"

    stats = {
        "total_deployments":    total,
        "verified_on_sourcify": verified,
        "verification_rate":    f"{ratio:.0%}",
        "first_deploy_date":    first_deploy,
        "last_deploy_date":     last_deploy,
        "active_years":         f"{deploy_span_days/365:.1f} years",
        "exact_matches":        int((df["overallMatch"] == "exact_match").sum()) if not df.empty else 0,
        "contracts_with_docs":  int(df["hasDevDoc"].sum()) if not df.empty else 0,
        "proxy_contracts":      int(df["isProxy"].sum()) if not df.empty else 0,
    }

    return {
        "deployer":          deployer_address,
        "total_deployed":    total,
        "verified":          verified,
        "reliability_score": score,
        "verdict":           verdict,
        "stats":             stats,
        "df":                df
    }


# ── ВЫВОД ──────────────────────────────────────────────────────────────
result = audit_developer(TEST_ADDRESS)

print("\n" + "=" * 60)
print("   DEVELOPER TRUST REPORT  |  powered by Sourcify")
print("=" * 60)
print(f"  Wallet:             {result['deployer']}")
print(f"  Trust Score:        {result['reliability_score']} / 1.0")
print(f"  Decision:           {result['verdict']}")
print("\n  ── Deployment Statistics ──")
for k, v in result["stats"].items():
    print(f"  {k:<28} {v}")

if not result["df"].empty:
    print("\n  ── Verified Contracts Detail ──")
    cols = ["address", "overallMatch", "name", "isProxy", "hasStorageLayout", "hasDevDoc", "sourceFileCount"]
    print(result["df"][cols].to_string(index=False))



Auditing wallet: 0xAb5801a7D398351b8bE11C439e05C5B3259aeC9B

Etherscan: 1 | OK
Deployments found: 0

   DEVELOPER TRUST REPORT  |  powered by Sourcify
  Wallet:             0xAb5801a7D398351b8bE11C439e05C5B3259aeC9B
  Trust Score:        0.0 / 1.0
  Decision:           REJECT — No deployment history

  ── Deployment Statistics ──


In [7]:

def explore_sourcify_fields(contract_address: str, chain_id: int = 1):
    """Показывает ВСЕ поля которые реально возвращает Sourcify для контракта"""
    url = f"https://sourcify.dev/server/v2/contract/{chain_id}/{contract_address}?fields=all"
    res = requests.get(url, timeout=40)
    
    if res.status_code != 200:
        print(f"❌ {contract_address} — не найден в Sourcify (статус {res.status_code})")
        return None
    
    data = res.json()
    
    print(f"\n{'='*60}")
    print(f"Контракт: {contract_address}")
    print(f"{'='*60}")
    
    # Показываем все ключи верхнего уровня
    print("\n📋 ВЕРХНИЙ УРОВЕНЬ (все ключи):")
    for key, value in data.items():
        if isinstance(value, dict):
            print(f"  {key}: {{dict, {len(value)} ключей}} → {list(value.keys())[:5]}")
        elif isinstance(value, list):
            print(f"  {key}: [list, {len(value)} элементов]")
        elif isinstance(value, str) and len(value) > 80:
            print(f"  {key}: '{value[:80]}...'")
        else:
            print(f"  {key}: {value}")
    
    # Разбираем compilation отдельно
    if "compilation" in data:
        print("\n📦 COMPILATION (подробно):")
        for key, value in data["compilation"].items():
            if isinstance(value, dict):
                print(f"  compilation.{key}: {{dict}} → {list(value.keys())[:8]}")
            elif isinstance(value, list):
                print(f"  compilation.{key}: [list, {len(value)} элементов]")
            elif isinstance(value, str) and len(value) > 80:
                print(f"  compilation.{key}: '{value[:80]}...'")
            else:
                print(f"  compilation.{key}: {value}")
    
    # Смотрим storageLayout если есть
    if data.get("storageLayout"):
        sl = data["storageLayout"]
        storage_vars = sl.get("storage", [])
        print(f"\n🗄️  STORAGE LAYOUT ({len(storage_vars)} переменных):")
        for var in storage_vars[:5]:
            print(f"  slot {var.get('slot')}: {var.get('type')} {var.get('label')}")
    
    # Смотрим ABI если есть
    if data.get("abi"):
        abi = data["abi"]
        functions = [x for x in abi if x.get("type") == "function"]
        events    = [x for x in abi if x.get("type") == "event"]
        errors    = [x for x in abi if x.get("type") == "error"]
        print(f"\n📐 ABI: {len(functions)} функций, {len(events)} событий, {len(errors)} ошибок")
        print("  Функции:", [f.get("name") for f in functions[:10]])
    
    # Смотрим devdoc
    if data.get("devdoc"):
        methods = data["devdoc"].get("methods", {})
        print(f"\n📝 DEVDOC: {len(methods)} задокументированных методов")
        for method_name in list(methods.keys())[:3]:
            print(f"  {method_name}")
    
    return data

# ── ТЕСТОВЫЕ АДРЕСА — разные типы разработчиков ──────────────
test_contracts = {
    "Uniswap Timelock (надёжный)":    "0x1a9C8182C09F50C8318d769245beA52c32BE35BC",
    "DAI стейблкоин (классика)":      "0x6B175474E89094C44Da98b954EedeAC495271d0F",
    "OpenZeppelin Proxy (продвинутый)":"0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48",
    "Chainlink Oracle":               "0x5f4ec3df9cbd43714fe2740f5e3616155c5b8419",
}

for name, addr in test_contracts.items():
    print(f"\n🔍 Проверяем: {name}")
    explore_sourcify_fields(addr)


🔍 Проверяем: Uniswap Timelock (надёжный)

Контракт: 0x1a9C8182C09F50C8318d769245beA52c32BE35BC

📋 ВЕРХНИЙ УРОВЕНЬ (все ключи):
  matchId: 2136710
  creationMatch: match
  runtimeMatch: match
  verifiedAt: 2024-08-08T14:40:04Z
  creationBytecode: {dict, 7 ключей} → ['onchainBytecode', 'recompiledBytecode', 'sourceMap', 'linkReferences', 'cborAuxdata']
  runtimeBytecode: {dict, 8 ключей} → ['onchainBytecode', 'recompiledBytecode', 'sourceMap', 'linkReferences', 'cborAuxdata']
  deployment: {dict, 4 ключей} → ['transactionHash', 'blockNumber', 'transactionIndex', 'deployer']
  sources: {dict, 1 ключей} → ['Timelock.sol']
  compilation: {dict, 6 ключей} → ['language', 'compiler', 'compilerVersion', 'compilerSettings', 'name']
  abi: [list, 21 элементов]
  metadata: {dict, 6 ключей} → ['compiler', 'language', 'output', 'settings', 'sources']
  storageLayout: {dict, 2 ключей} → ['types', 'storage']
  transientStorageLayout: None
  userdoc: {dict, 1 ключей} → ['methods']
  devdoc: {dict, 1 к

In [11]:
from datetime import datetime, timezone
import time

test_deployers = {
    # 🟢 Ожидаем высокий скор
    "Uniswap Labs deployer":     "0x1a9C8182C09F50C8318d769245beA52c32BE35BC",
    "Aave deployer":             "0xEE56e2B3D491590B5b31738cC34d5232F378a8D5",
    "OpenZeppelin deployer":     "0x9B397f841e1dce5C32A7028D6CD51B88f5E3A490",

    # 🟡 Ожидаем средний скор  
    "Небольшой проект":          "0x3ea56dea75abed066bb679e61469fd1f37102139",

    # 🔴 Ожидаем нулевой скор
    "Виталик (не деплоер)":      "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
    "Случайный кошелёк":         "0xAb5801a7D398351b8bE11C439e05C5B3259aeC9B",
}

for name, addr in test_deployers.items():
    print(f"\n{'='*50}")
    print(f"Тест: {name}")
    result = audit_developer(addr)
    print(f"Скор: {result['reliability_score']} | {result['verdict']}")


Тест: Uniswap Labs deployer

Auditing wallet: 0x1a9C8182C09F50C8318d769245beA52c32BE35BC

Etherscan: 1 | OK
Deployments found: 1
Checking on Sourcify...


/var/folders/x8/4p5hc8sn2rld4r2vqq1f007h0000gn/T/ipykernel_46537/1264663363.py:109: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  first_deploy = datetime.utcfromtimestamp(min(timestamps)).strftime("%Y-%m-%d")
/var/folders/x8/4p5hc8sn2rld4r2vqq1f007h0000gn/T/ipykernel_46537/1264663363.py:110: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  last_deploy  = datetime.utcfromtimestamp(max(timestamps)).strftime("%Y-%m-%d")


  ✓ 0x1a9c8182c0... | match | Timelock | proxy:False | docs:False | files:1
Скор: 0.55 | CONDITIONAL — Open with stricter KPI

Тест: Aave deployer

Auditing wallet: 0xEE56e2B3D491590B5b31738cC34d5232F378a8D5

Etherscan: 1 | OK
Deployments found: 1
Checking on Sourcify...


/var/folders/x8/4p5hc8sn2rld4r2vqq1f007h0000gn/T/ipykernel_46537/1264663363.py:109: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  first_deploy = datetime.utcfromtimestamp(min(timestamps)).strftime("%Y-%m-%d")
/var/folders/x8/4p5hc8sn2rld4r2vqq1f007h0000gn/T/ipykernel_46537/1264663363.py:110: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  last_deploy  = datetime.utcfromtimestamp(max(timestamps)).strftime("%Y-%m-%d")


  ✓ 0xee56e2b3d4... | match | Executor | proxy:False | docs:True | files:1
Скор: 0.6 | CONDITIONAL — Open with stricter KPI

Тест: OpenZeppelin deployer

Auditing wallet: 0x9B397f841e1dce5C32A7028D6CD51B88f5E3A490

Etherscan: 0 | No transactions found
Скор: 0.0 | REJECT — No deployment history

Тест: Небольшой проект

Auditing wallet: 0x3ea56dea75abed066bb679e61469fd1f37102139

Etherscan: 1 | OK
Deployments found: 10
Checking on Sourcify...


/var/folders/x8/4p5hc8sn2rld4r2vqq1f007h0000gn/T/ipykernel_46537/1264663363.py:109: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  first_deploy = datetime.utcfromtimestamp(min(timestamps)).strftime("%Y-%m-%d")
/var/folders/x8/4p5hc8sn2rld4r2vqq1f007h0000gn/T/ipykernel_46537/1264663363.py:110: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  last_deploy  = datetime.utcfromtimestamp(max(timestamps)).strftime("%Y-%m-%d")


Warning 0xd92d8c9c: 'NoneType' object has no attribute 'get'
Warning 0xe3b38a52: 'NoneType' object has no attribute 'get'
  ✓ 0x00171eda11... | match | SafeMathLibExt | proxy:False | docs:False | files:1
Скор: 0.36 | REJECT — Low trust score

Тест: Виталик (не деплоер)

Auditing wallet: 0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045

Etherscan: 1 | OK
Deployments found: 354
Checking on Sourcify...


/var/folders/x8/4p5hc8sn2rld4r2vqq1f007h0000gn/T/ipykernel_46537/1264663363.py:109: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  first_deploy = datetime.utcfromtimestamp(min(timestamps)).strftime("%Y-%m-%d")
/var/folders/x8/4p5hc8sn2rld4r2vqq1f007h0000gn/T/ipykernel_46537/1264663363.py:110: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  last_deploy  = datetime.utcfromtimestamp(max(timestamps)).strftime("%Y-%m-%d")


Скор: 0.02 | REJECT — Low trust score

Тест: Случайный кошелёк

Auditing wallet: 0xAb5801a7D398351b8bE11C439e05C5B3259aeC9B

Etherscan: 1 | OK
Deployments found: 0
Скор: 0.0 | REJECT — No deployment history
